# Scotland Weather Forecast Synthesiser

This notebook scrapes several specialist forecast sources for Scotland and uses GPT to synthesise them into a single, clear summary covering:

- **Snow & Mountain conditions** (Highlands & Cairngorms)
- **Surf** (Thurso East — Scotland's most famous wave)
- **Rain & General Weather**
- **Wind**

Run each cell top-to-bottom with `Shift+Enter`.

> **Note:** Some weather sites use JavaScript rendering and may return limited content with this simple scraper. The notebook handles that gracefully — if a source returns little data the LLM will say so.

In [14]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from playwright.async_api import async_playwright

In [15]:
# Load environment variables and check the key

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key found - check your .env file!")
elif not api_key.startswith("sk-proj-"):
    print("API key found but doesn't start with sk-proj- — double-check it.")
else:
    print("API key found and looks good!")

API key found and looks good!


In [16]:
openai = OpenAI()

## Scraper

We use **Playwright** (a headless Chromium browser) instead of plain `requests` so that JavaScript-rendered pages — like surf forecast sites and Met Office — fully load before we extract text.

One browser instance is shared across all fetches for speed.

In [17]:
import requests as _requests

SCOTTISH_BUOYS = ['torness', 'arbroath', 'buckie', 'orkney', 'stonehaven', 'eyemouth']

def fetch_wavenet_buoys():
    """Pull live readings for Scottish wave buoys straight from the WaveNet REST API."""
    try:
        r = _requests.get('https://wavenet-api.cefas.co.uk/api/Map/Current', timeout=10)
        features = r.json()['features']
        lines = ["WaveNet Live Scottish Buoy Readings\n"]
        for f in features:
            title = f['properties']['title']
            if any(s in title.lower() for s in SCOTTISH_BUOYS):
                p = f['properties']
                results = p.get('results', {})
                lines.append(f"\n{title}  (as of {p['timestamp']})")
                for v in results.values():
                    val = v['values'][0] if v['values'] else ''
                    if val:
                        lines.append(f"  {v['description']}: {val} {v['unit']}")
        return '\n'.join(lines)
    except Exception as e:
        return f"[Could not fetch WaveNet data: {e}]"


async def fetch_forecast(page, url, char_limit=5_000):
    """Render a page with Playwright, dismiss cookie consent, and return visible text."""
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=30_000)
        await page.wait_for_timeout(1500)
        # Dismiss common consent banners
        for selector in ['button:has-text("AGREE")', 'button:has-text("Accept")',
                         'button:has-text("I agree")', 'button:has-text("Accept all")']:
            try:
                await page.click(selector, timeout=1500)
                await page.wait_for_timeout(800)
                break
            except:
                pass
        await page.evaluate("""
            document.querySelectorAll('script,style,nav,footer,header,img,input,iframe')
                     .forEach(el => el.remove())
        """)
        title = await page.title()
        body  = await page.inner_text("body")
        return (title + "\n\n" + body)[:char_limit]
    except Exception as e:
        return f"[Could not fetch {url}: {e}]"

"## Forecast Sources\n",
    "\n",
    "| Source | Covers |\n",
    "|--------|--------|\n",
    "| MWIS Highlands & Cairngorms | Snow, mountain weather |\n",
    "| MWIS Western Highlands | Snow, mountain weather |\n",
    "| XCWeather Scotland | Wind, rain, general |\n",
    "| Met Office Scotland overview | Rain, wind, general |\n",
    "| **Surf** | Chosen below — pick as many spots as you like |"

In [18]:
BASE_SOURCES = {
    "Mountain & Snow — Cairngorms & Monadhliath (MWIS)":
        "https://www.mwis.org.uk/forecasts/scottish/cairngorms-np-and-monadhliath/text",
    "Mountain & Snow — West Highlands incl. Ben Nevis & Glencoe (MWIS)":
        "https://www.mwis.org.uk/forecasts/scottish/west-highlands/text",
    "Mountain & Snow — Northwest Highlands (MWIS)":
        "https://www.mwis.org.uk/forecasts/scottish/the-northwest-highlands/text",
    "Wind & Rain — Scotland (XCWeather)":
        "https://www.xcweather.co.uk/forecast/scotland",
    "General — Met Office Scotland":
        "https://www.metoffice.gov.uk/weather/forecast/gcpvj0v07",
}

## Choose Your Surf Spots\n\nAll known Scottish surf breaks are listed in `SURF_SPOTS` below.\nEdit `selected_surf_spots` to include whichever ones you want in the briefing.

In [19]:
# Available Scottish surf spots on surf-forecast.com
# Note: Belhaven Bay is not listed on surf-forecast.com — use the Torness WaveNet buoy
# (fetched separately below) for live East Lothian sea conditions.
SURF_SPOTS = {
    "Thurso East":      "https://www.surf-forecast.com/breaks/Thurso-East/forecasts/latest",
    "Pease Bay":        "https://www.surf-forecast.com/breaks/Pease-Bay/forecasts/latest",
    "Machrihanish":     "https://www.surf-forecast.com/breaks/Machrihanish/forecasts/latest",
    "Scarista":         "https://www.surf-forecast.com/breaks/Scarista/forecasts/latest",
    "Tiree (The Maze)": "https://www.surf-forecast.com/breaks/The-Maze/forecasts/latest",
    "Coldingham Bay":   "https://www.surf-forecast.com/breaks/Coldingham-Bay/forecasts/latest",
    "Sandend Bay":      "https://www.surf-forecast.com/breaks/Sandend-Bay/forecasts/latest",
}

print("Available surf spots:")
for name in SURF_SPOTS:
    print(f"  - {name}")
print()
print("ℹ️  Belhaven Bay conditions are covered by the Torness WaveNet buoy (live sea state).")

Available surf spots:
  - Thurso East
  - Pease Bay
  - Machrihanish
  - Scarista
  - Tiree (The Maze)
  - Coldingham Bay
  - Sandend Bay

ℹ️  Belhaven Bay conditions are covered by the Torness WaveNet buoy (live sea state).


In [20]:
# ✏️  Edit this list to choose which surf spots to include
selected_surf_spots = [
    "Belhaven Bay",
]

# Build the final forecast sources dict from base sources + selected surf spots
FORECAST_SOURCES = dict(BASE_SOURCES)
for spot in selected_surf_spots:
    if spot in SURF_SPOTS:
        FORECAST_SOURCES[f"Surf — {spot}"] = SURF_SPOTS[spot]
    else:
        print(f"⚠️  '{spot}' not found in SURF_SPOTS — check the name and try again.")

print("Forecast sources for this run:")
for name in FORECAST_SOURCES:
    print(f"  • {name}")

⚠️  'Belhaven Bay' not found in SURF_SPOTS — check the name and try again.
Forecast sources for this run:
  • Mountain & Snow — Cairngorms & Monadhliath (MWIS)
  • Mountain & Snow — West Highlands incl. Ben Nevis & Glencoe (MWIS)
  • Mountain & Snow — Northwest Highlands (MWIS)
  • Wind & Rain — Scotland (XCWeather)
  • General — Met Office Scotland


## Fetch All Forecasts

Opens one headless Chromium browser, visits each source in turn, and extracts the rendered text. Expect this to take 20–40 seconds total.

In [21]:
raw_forecasts = {}

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page()

    for name, url in FORECAST_SOURCES.items():
        print(f"Fetching: {name}...")
        content = await fetch_forecast(page, url)
        raw_forecasts[name] = content
        print(f"  → {len(content):,} characters retrieved\n")

    await browser.close()

# WaveNet buoy data — direct API call, no browser needed
print("Fetching: Live sea state — Scottish WaveNet buoys...")
raw_forecasts["Live Sea State — Scottish WaveNet Buoys (incl. Torness / Belhaven Bay)"] = fetch_wavenet_buoys()
print(f"  → {len(raw_forecasts['Live Sea State — Scottish WaveNet Buoys (incl. Torness / Belhaven Bay)']):,} characters retrieved\n")

Fetching: Mountain & Snow — Cairngorms & Monadhliath (MWIS)...
  → 5,000 characters retrieved

Fetching: Mountain & Snow — West Highlands incl. Ben Nevis & Glencoe (MWIS)...
  → 5,000 characters retrieved

Fetching: Mountain & Snow — Northwest Highlands (MWIS)...
  → 5,000 characters retrieved

Fetching: Wind & Rain — Scotland (XCWeather)...
  → 5,000 characters retrieved

Fetching: General — Met Office Scotland...
  → 5,000 characters retrieved

Fetching: Live sea state — Scottish WaveNet buoys...
  → 1,389 characters retrieved



## Prompts

We give the model a system prompt that sets it up as a Scottish weather expert, then a user prompt that bundles all the raw forecast text for synthesis.

In [22]:
system_prompt = """
You are an expert Scottish weather analyst who knows the Highlands, the Cairngorms,
and the north coast surf scene well. You synthesise raw forecast data from multiple
sources into one clear, readable briefing.

Structure your response in markdown with these sections:
1. **Overview** — one-paragraph summary of the overall picture
2. **Snow & Mountain Conditions** — avalanche risk, snow depth, visibility, wind chill
3. **Surf** — swell height, period, wind direction, quality rating per spot
4. **Belhaven Bay / East Lothian Sea State** — use the Torness WaveNet buoy readings
   (significant wave height, peak period, wave direction) to describe current conditions
5. **Rain & Wind** — precipitation amounts, wind speeds and directions, any warnings
6. **Outlook** — brief 3-5 day trend

If a section has no usable data, say so briefly rather than making things up.
Respond in markdown. Do not wrap the markdown in a code block.
"""

In [23]:
def build_user_prompt(forecasts):
    prompt = "Please synthesise the following raw forecast data for Scotland into a structured briefing.\n\n"
    for source_name, content in forecasts.items():
        prompt += f"---\n### Source: {source_name}\n{content}\n\n"
    return prompt

## Synthesise!

Now we pass all the fetched content to GPT and ask it to produce the unified forecast briefing.

In [24]:
def synthesise_forecasts(forecasts):
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": build_user_prompt(forecasts)},
        ]
    )
    return response.choices[0].message.content

In [25]:
briefing = synthesise_forecasts(raw_forecasts)
display(Markdown(briefing))

# Scotland Weather Briefing

## Overview
A notable cold snap is set to impact much of Scotland, particularly in the Highlands and coastal areas. Expect cool temperatures, gusty winds, and significant wind chill across mountainous regions. Snow and hail showers are likely particularly in northern and eastern areas as conditions fluctuate from clear to overcast.

## Snow & Mountain Conditions
- **Avalanche Risk**: Generally low, but varied depending on exposure and recent snow accumulation; caution is advised in steep terrain.
- **Snow Depth**: Snow is anticipated mainly at higher elevations, with conditions improving midweek.
- **Visibility**: Rapid changes from excellent to very poor due to winter showers are expected, especially in the mornings.
- **Wind Chill**: At 900m, temperatures are forecasted around -3°C, feeling like -10°C to -15°C under sustained wind.

## Surf
- **Swell Height**: 1.24 m at Torness, indicative of moderate surf conditions.
- **Period**: Dominant wave period at 4.7 seconds; surf is relatively choppy.
- **Wind Direction**: Offering moderate onshore winds which could affect wave quality.
- **Quality Rating**: Mixed quality, with best conditions likely seen early mornings; suitable for experienced surfers.

## Belhaven Bay / East Lothian Sea State
As of the latest report from the Torness WaveNet buoy:
- **Significant Wave Height**: 1.24 m
- **Peak Period**: 4.7 seconds
- **Wave Direction**: 14°, indicating incoming swells from the east; surf conditions are moderate with potential for choppy waves.

## Rain & Wind
- **Precipitation Amounts**: Minimal rain expected on Monday, shifting to more substantial snow/sleet on Tuesday, especially in southern areas.
- **Wind Speeds and Directions**: Winds initially from the north at 15-25 mph, shifting to gusts of 50-70 mph from the southeast later in the week, with severe gales expected post-dark.
- **Warnings**: A yellow weather warning for wind is likely needed as conditions deteriorate this week, particularly in coastal and exposed areas.

## Outlook
- **Monday (16th)**: Snow showers early, becoming mostly dry and sunny by afternoon.
- **Tuesday (17th)**: Transition to persistent hill snow with difficult conditions due to increasing easterly winds.
- **Wednesday (18th)**: Continued cold with snow flurries; lifting slightly in temperature, but wind chill remains a factor.
- **Thursday to Saturday (19–21)**: Gradually improving temperatures but retaining cool, breezy conditions with potential light snow.

## Try It Yourself

Some ideas to extend this:
- Add a source for sea temperature (useful for surfers and wild swimmers)
- Change the system prompt to output a **WhatsApp-style** message instead of markdown
- Add a specific location (e.g. Ben Nevis, Glencoe, Caithness) and tailor the briefing
- Use `gpt-4o` instead of `gpt-4o-mini` for a more detailed synthesis

In [26]:
# Your turn — add extra sources or tweak the system prompt here

# Example: add a sea temperature source
# extra_sources = {
#     "Sea Temp — Scotland": "https://www.seatemperature.org/europe/united-kingdom/scotland.htm",
# }
# for name, url in extra_sources.items():
#     raw_forecasts[name] = fetch_forecast(url)

# Then re-run: display(Markdown(synthesise_forecasts(raw_forecasts)))